# 27 — Final Evaluation

This notebook consolidates the evaluation artifacts produced by the Adaptive Road Vision project.

It evaluates the project at **component level** and **system level**:

1. Dataset / split integrity
2. Condition classification
3. Enhancement
4. Detector comparison and selected detector
5. Original vs enhanced detection
6. ByteTrack operational evaluation
7. Monocular depth estimation
8. Risk engine
9. End-to-end pipeline
10. Final consolidated report

### Important interpretation rules
- This notebook **reads existing evaluation artifacts**; it does not retrain models.
- Test-set results are treated as the final held-out evidence where available.
- Missing artifacts are reported rather than fabricated.
- ByteTrack is evaluated operationally because object-level tracking ground truth is not available in the current project metadata.
- Depth is relative unless calibrated metric scale is explicitly supplied.
- Enhancement PSNR/SSIM use the project's pseudo-reference targets, not photographic ground truth.
- The risk score is an interpretable project score, not collision probability or a safety certification.
- Runtime/FPS is reported only when measured by the project artifacts.

In [ ]:
from IPython.display import display
from pathlib import Path
import json, os, math, warnings, re
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# Prefer an explicit environment variable; otherwise check common Windows locations.
# If none exists, edit PROJECT_ROOT below to your actual project directory.
_candidates = []
_env_root = os.environ.get("ADAPTIVE_ROAD_VISION_ROOT")
if _env_root:
    _candidates.append(Path(_env_root).expanduser())
_candidates.extend([
    Path(r"C:\Users\ankit\Desktop\Adaptive_Road_Vision"),
    Path(r"C:\Users\CUTM\Desktop\Adaptive_Road_Vision"),
    Path.home() / "Desktop" / "Adaptive_Road_Vision",
    Path.cwd() / "Adaptive_Road_Vision",
    Path.cwd(),
])
PROJECT_ROOT = next((p.resolve() for p in _candidates if p.exists() and (p / "results").exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate Adaptive_Road_Vision project/results folder. "
        "Set ADAPTIVE_ROAD_VISION_ROOT or edit PROJECT_ROOT in this cell to your project folder. "
        f"Checked: {[str(p) for p in _candidates]}"
    )

RESULTS = PROJECT_ROOT / "results"
OUT = RESULTS / "final_evaluation"
OUT.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_ROOT)
print("Final evaluation output:", OUT)


## 1. Artifact inventory

In [ ]:
ARTIFACTS = {
    "dataset_summary": next((p for p in [
        RESULTS / "dataset" / "final_dataset_summary.json",
        RESULTS / "dataset_balance_audit_v2" / "final_dataset_build_report.json",
        RESULTS / "dataset_balance_audit" / "final_dataset_build_report.json"
    ] if p.exists()), RESULTS / "dataset" / "final_dataset_summary.json"),

    "condition_report": RESULTS / "condition" / "condition_evaluation_report.json",
    "condition_predictions": RESULTS / "condition" / "condition_test_predictions.csv",

    "enhancement_summary": RESULTS / "enhancement" / "enhancement_evaluation_summary.csv",
    "enhancement_report": RESULTS / "enhancement" / "enhancement_evaluation_report.json",
    "enhancement_per_image": RESULTS / "enhancement" / "enhancement_evaluation_per_image.csv",

    "detector_comparison": RESULTS / "detection" / "detector_comparison_summary.csv",
    "detector_comparison_report": RESULTS / "detection" / "detector_comparison_report.json",
    "detector_selection": RESULTS / "detection" / "final_detector_selection.json",

    "original_vs_enhanced": RESULTS / "detection" / "original_vs_enhanced_detection_overall.csv",
    "original_vs_enhanced_condition": RESULTS / "detection" / "original_vs_enhanced_detection_by_condition.csv",
    "original_vs_enhanced_report": RESULTS / "detection" / "original_vs_enhanced_detection_report.json",

    "tracking_overall": RESULTS / "tracking" / "bytetrack_overall_metrics.csv",
    "tracking_sequence": RESULTS / "tracking" / "bytetrack_sequence_metrics.csv",
    "tracking_report": RESULTS / "tracking" / "bytetrack_evaluation_report.json",

    "depth_objects": RESULTS / "depth" / "depth_object_estimates.csv",
    "depth_condition": RESULTS / "depth" / "depth_condition_summary.csv",
    "depth_report": RESULTS / "depth" / "depth_estimation_report.json",

    "risk_frames": RESULTS / "risk" / "risk_frame_scores.csv",
    "risk_condition": RESULTS / "risk" / "risk_condition_summary.csv",
    "risk_levels": RESULTS / "risk" / "risk_level_distribution.csv",
    "risk_report": RESULTS / "risk" / "risk_engine_report.json",

    "pipeline_frames": RESULTS / "pipeline" / "pipeline_per_frame.csv",
    "pipeline_objects": RESULTS / "pipeline" / "pipeline_object_details.csv",
    "pipeline_condition": RESULTS / "pipeline" / "pipeline_condition_summary.csv",
    "pipeline_risk": RESULTS / "pipeline" / "pipeline_risk_summary.csv",
    "pipeline_report": RESULTS / "pipeline" / "end_to_end_pipeline_report.json",
}

status_rows = []
for name, path in ARTIFACTS.items():
    status_rows.append({
        "artifact": name,
        "exists": path.exists(),
        "path": str(path),
        "size_bytes": path.stat().st_size if path.exists() else None,
    })

artifact_status = pd.DataFrame(status_rows)
display(artifact_status)

artifact_status.to_csv(OUT / "final_evaluation_artifact_status.csv", index=False)


## 2. Helper functions for safe artifact loading

In [ ]:
def load_json(path):
    if not path.exists():
        return None
    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception as e:
        print("JSON read warning:", path, e)
        return None

def load_csv(path):
    if not path.exists():
        return None
    try:
        return pd.read_csv(path)
    except Exception as e:
        print("CSV read warning:", path, e)
        return None

def first_existing(*paths):
    for p in paths:
        if p is not None and Path(p).exists():
            return Path(p)
    return None

def get_number(d, *keys):
    if not isinstance(d, dict):
        return None
    for k in keys:
        if k in d:
            try:
                return float(d[k])
            except Exception:
                pass
    return None

## 3. Dataset integrity

In [ ]:
dataset_report = load_json(ARTIFACTS["dataset_summary"])

dataset_rows = []
if isinstance(dataset_report, dict):
    def flatten(obj, prefix=""):
        rows = []
        if isinstance(obj, dict):
            for k, v in obj.items():
                key = f"{prefix}.{k}" if prefix else str(k)
                if isinstance(v, (dict, list)):
                    rows.extend(flatten(v, key))
                else:
                    rows.append((key, v))
        elif isinstance(obj, list):
            rows.append((prefix, obj))
        return rows

    dataset_rows = flatten(dataset_report)

dataset_table = pd.DataFrame(dataset_rows, columns=["metric", "value"])
display(dataset_table.head(50))

print("Dataset summary available:", dataset_report is not None)

## 4. Condition classification — final test evaluation

In [ ]:
try:
    from IPython.display import display
except ImportError:
    pass

def show(obj):
    try:
        from IPython.display import display as _disp
        _disp(obj)
    except Exception:
        if callable(globals().get("display")):
            display(obj)
        elif hasattr(globals().get("display"), "display") and callable(globals().get("display").display):
            globals().get("display").display(obj)
        else:
            print(obj)

condition_report = load_json(ARTIFACTS["condition_report"])
condition_predictions = load_csv(ARTIFACTS["condition_predictions"])

condition_metrics = []

# 1. Search condition report: check nested 'test', 'test_metrics', 'metrics', or top-level keys
if isinstance(condition_report, dict):
    test_dict = None
    for subkey in ["test", "test_metrics", "metrics"]:
        if isinstance(condition_report.get(subkey), dict):
            test_dict = condition_report[subkey]
            break
    if test_dict is None:
        test_dict = condition_report

    metric_keys = [
        "samples", "test_samples",
        "accuracy", "test_accuracy",
        "balanced_accuracy", "test_balanced_accuracy",
        "macro_precision", "test_macro_precision",
        "macro_recall", "test_macro_recall",
        "macro_f1", "test_macro_f1",
    ]
    for k in metric_keys:
        if k in test_dict:
            condition_metrics.append({"metric": k, "value": test_dict[k]})
        elif k in condition_report:
            condition_metrics.append({"metric": k, "value": condition_report[k]})

# 2. Fallback to condition_metrics_summary.csv if available
cond_summary_path = RESULTS / "condition" / "condition_metrics_summary.csv"
if not condition_metrics and cond_summary_path.exists():
    summary_df = load_csv(cond_summary_path)
    if summary_df is not None and not summary_df.empty:
        test_row = summary_df[summary_df["split"].astype(str).str.lower() == "test"]
        if test_row.empty:
            test_row = summary_df.iloc[[-1]]
        for col in ["samples", "accuracy", "balanced_accuracy", "macro_precision", "macro_recall", "macro_f1"]:
            if col in test_row.columns:
                condition_metrics.append({"metric": col, "value": float(test_row.iloc[0][col])})

# 3. Fallback to computing directly from condition_predictions DataFrame
if not condition_metrics and condition_predictions is not None and not condition_predictions.empty:
    if "correct" in condition_predictions.columns:
        acc = float(condition_predictions["correct"].mean())
    elif "true_condition" in condition_predictions.columns and "predicted_condition" in condition_predictions.columns:
        acc = float((condition_predictions["true_condition"] == condition_predictions["predicted_condition"]).mean())
    else:
        acc = None

    if acc is not None:
        condition_metrics.append({"metric": "accuracy", "value": acc})
        condition_metrics.append({"metric": "samples", "value": len(condition_predictions)})

    if "true_condition" in condition_predictions.columns and "predicted_condition" in condition_predictions.columns:
        classes = sorted(condition_predictions["true_condition"].dropna().unique())
        recalls, precisions, f1s = [], [], []
        for c in classes:
            tp = int(((condition_predictions["true_condition"] == c) & (condition_predictions["predicted_condition"] == c)).sum())
            fn = int(((condition_predictions["true_condition"] == c) & (condition_predictions["predicted_condition"] != c)).sum())
            fp = int(((condition_predictions["true_condition"] != c) & (condition_predictions["predicted_condition"] == c)).sum())
            rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
            prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
            f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
            recalls.append(rec)
            precisions.append(prec)
            f1s.append(f1)
        if recalls:
            condition_metrics.append({"metric": "balanced_accuracy", "value": float(np.mean(recalls))})
            condition_metrics.append({"metric": "macro_precision", "value": float(np.mean(precisions))})
            condition_metrics.append({"metric": "macro_recall", "value": float(np.mean(recalls))})
            condition_metrics.append({"metric": "macro_f1", "value": float(np.mean(f1s))})

if condition_metrics:
    condition_metrics_df = pd.DataFrame(condition_metrics).drop_duplicates("metric")
    print("=== Condition Test Evaluation Metrics ===")
    show(condition_metrics_df)
else:
    condition_metrics_df = pd.DataFrame(columns=["metric", "value"])
    print("No condition metrics were found or computed.")

# Display per-class metrics if available
per_class_df = None
if isinstance(condition_report, dict) and "test" in condition_report and "per_class" in condition_report["test"]:
    per_class_df = pd.DataFrame(condition_report["test"]["per_class"])
elif (RESULTS / "condition" / "condition_test_per_class_metrics.csv").exists():
    per_class_df = load_csv(RESULTS / "condition" / "condition_test_per_class_metrics.csv")

if per_class_df is not None and not per_class_df.empty:
    print("\n=== Per-Class Performance ===")
    show(per_class_df)

# Display confusion matrix if available
cm_path = first_existing(
    RESULTS / "condition" / "condition_confusion_matrix.csv",
    RESULTS / "condition" / "condition_test_confusion_matrix_raw.csv"
)
if cm_path:
    cm_df = load_csv(cm_path)
    if cm_df is not None and not cm_df.empty:
        print("\n=== Confusion Matrix ===")
        show(cm_df)

if condition_predictions is not None and not condition_predictions.empty:
    print(f"\nCondition test predictions: {len(condition_predictions):,}")
    print("Prediction columns:", list(condition_predictions.columns))


## 5. Enhancement evaluation

In [ ]:
enhancement_summary = load_csv(ARTIFACTS["enhancement_summary"])
enhancement_report = load_json(ARTIFACTS["enhancement_report"])

if enhancement_summary is not None:
    display(enhancement_summary)

if isinstance(enhancement_report, dict):
    print("Enhancement report keys:", list(enhancement_report.keys()))
else:
    print("Enhancement report unavailable.")

## 6. Detector comparison and selected detector

In [ ]:
detector_comparison = load_csv(ARTIFACTS["detector_comparison"])
detector_selection = load_json(ARTIFACTS["detector_selection"])

if detector_comparison is not None:
    display(detector_comparison)

print("\nFinal detector selection artifact:")
if detector_selection is not None:
    display(pd.DataFrame(
        [(k, v) for k, v in detector_selection.items()],
        columns=["field", "value"]
    ))
else:
    print("Selection artifact unavailable.")

# Extract the selected detector name without assigning a new ranking here.
selected_detector = None
if isinstance(detector_selection, dict):
    for key in ["selected_detector", "model", "detector_name", "selected_model"]:
        if key in detector_selection:
            selected_detector = detector_selection[key]
            break

print("Selected detector recorded by Notebook 23:", selected_detector)

## 7. Original vs enhanced detection

In [ ]:
det_overall = load_csv(ARTIFACTS["original_vs_enhanced"])
det_by_condition = load_csv(ARTIFACTS["original_vs_enhanced_condition"])
det_report = load_json(ARTIFACTS["original_vs_enhanced_report"])

if det_overall is not None:
    print("Overall detection comparison:")
    display(det_overall)

if det_by_condition is not None:
    print("Detection comparison by condition:")
    display(det_by_condition)

if isinstance(det_report, dict):
    print("Detection report keys:", list(det_report.keys()))

## 8. ByteTrack operational evaluation

In [ ]:
tracking_overall = load_csv(ARTIFACTS["tracking_overall"])
tracking_sequence = load_csv(ARTIFACTS["tracking_sequence"])
tracking_report = load_json(ARTIFACTS["tracking_report"])

if tracking_overall is not None:
    display(tracking_overall)

if tracking_sequence is not None:
    print("Sequence-level tracking diagnostics:")
    display(tracking_sequence.head(20))

if isinstance(tracking_report, dict):
    print("Tracking report notes:")
    for note in tracking_report.get("notes", []):
        print("-", note)

## 9. Depth estimation

In [ ]:
depth_objects = load_csv(ARTIFACTS["depth_objects"])
depth_condition = load_csv(ARTIFACTS["depth_condition"])
depth_report = load_json(ARTIFACTS["depth_report"])

if depth_condition is not None:
    display(depth_condition)

if depth_objects is not None:
    print("Depth object estimates:", len(depth_objects))
    display(depth_objects.head(10))

if isinstance(depth_report, dict):
    print("Depth report:")
    for k, v in depth_report.items():
        if k != "notes":
            print(f"{k}: {v}")
    if depth_report.get("notes"):
        print("Notes:")
        for note in depth_report["notes"]:
            print("-", note)

## 10. Risk engine

In [ ]:
risk_frames = load_csv(ARTIFACTS["risk_frames"])
risk_condition = load_csv(ARTIFACTS["risk_condition"])
risk_levels = load_csv(ARTIFACTS["risk_levels"])
risk_report = load_json(ARTIFACTS["risk_report"])

if risk_condition is not None:
    print("Risk by condition:")
    display(risk_condition)

if risk_levels is not None:
    print("Risk level distribution:")
    display(risk_levels)

if risk_frames is not None and not risk_frames.empty:
    numeric_cols = [c for c in ["frame_risk", "risk_score", "score"] if c in risk_frames.columns]
    if numeric_cols:
        c = numeric_cols[0]
        print("Risk score descriptive statistics:")
        display(risk_frames[c].describe().to_frame("value"))

if isinstance(risk_report, dict):
    print("Risk report keys:", list(risk_report.keys()))

## 11. End-to-end pipeline evaluation

In [ ]:
pipeline_frames = load_csv(ARTIFACTS["pipeline_frames"])
pipeline_objects = load_csv(ARTIFACTS["pipeline_objects"])
pipeline_condition = load_csv(ARTIFACTS["pipeline_condition"])
pipeline_risk = load_csv(ARTIFACTS["pipeline_risk"])
pipeline_report = load_json(ARTIFACTS["pipeline_report"])

if pipeline_frames is not None and not pipeline_frames.empty:
    print("Pipeline frame summary:")
    display(pipeline_frames.head(10))

    numeric = {}
    for c in ["condition_confidence", "num_detections", "num_active_tracks",
              "frame_risk", "runtime_seconds", "measured_fps"]:
        if c in pipeline_frames.columns:
            numeric[c] = {
                "mean": float(pd.to_numeric(pipeline_frames[c], errors="coerce").mean()),
                "median": float(pd.to_numeric(pipeline_frames[c], errors="coerce").median()),
            }
    pipeline_numeric_summary = pd.DataFrame(numeric).T
    display(pipeline_numeric_summary)
else:
    pipeline_numeric_summary = pd.DataFrame()

if pipeline_condition is not None:
    print("Pipeline condition summary:")
    display(pipeline_condition)

if pipeline_risk is not None:
    print("Pipeline risk summary:")
    display(pipeline_risk)

if isinstance(pipeline_report, dict):
    print("Pipeline report:")
    for k in ["input_mode", "input_image_count", "processed_frame_count",
              "total_runtime_seconds", "measured_pipeline_fps",
              "device", "depth_enabled", "depth_model_id"]:
        if k in pipeline_report:
            print(f"{k}: {pipeline_report[k]}")

## 12. Build a consolidated metric table

In [ ]:
rows = []

def add_metric(component, metric, value, source):
    if value is None:
        return
    # Do not write NaN/Inf as if they were valid evaluation results.
    try:
        numeric_value = float(value)
        if not math.isfinite(numeric_value):
            return
        value = numeric_value
    except (TypeError, ValueError):
        pass
    rows.append({
        "component": component,
        "metric": metric,
        "value": value,
        "source_artifact": source,
    })

# Condition metrics
for _, r in condition_metrics_df.iterrows():
    add_metric("Condition", str(r["metric"]), r["value"], str(ARTIFACTS["condition_report"]))

# Detector metrics
if detector_comparison is not None and not detector_comparison.empty:
    # Keep the table descriptive. Do not create a new ranking.
    for col in ["mAP50", "mAP50-95", "precision", "recall", "map50", "map50_95"]:
        if col in detector_comparison.columns:
            vals = pd.to_numeric(detector_comparison[col], errors="coerce")
            add_metric(
                "Detector comparison",
                f"{col}_mean_across_reported_detectors",
                float(vals.mean()),
                str(ARTIFACTS["detector_comparison"])
            )

# Detection comparison
if det_overall is not None and not det_overall.empty:
    for col in ["precision", "recall", "f1", "F1"]:
        if col in det_overall.columns:
            vals = pd.to_numeric(det_overall[col], errors="coerce")
            add_metric(
                "Original vs enhanced detection",
                f"{col}_mean",
                float(vals.mean()),
                str(ARTIFACTS["original_vs_enhanced"])
            )

# Tracking
if tracking_overall is not None and not tracking_overall.empty:
    for col in ["mean_detections_per_frame", "mean_active_tracks",
                "mean_track_lifespan", "continuity_ratio", "mean_confidence"]:
        if col in tracking_overall.columns:
            vals = pd.to_numeric(tracking_overall[col], errors="coerce")
            add_metric(
                "ByteTrack",
                f"{col}_mean",
                float(vals.mean()),
                str(ARTIFACTS["tracking_overall"])
            )

# Pipeline
if pipeline_frames is not None and not pipeline_frames.empty:
    for col in ["condition_confidence", "num_detections", "num_active_tracks",
                "frame_risk", "measured_fps"]:
        if col in pipeline_frames.columns:
            vals = pd.to_numeric(pipeline_frames[col], errors="coerce")
            add_metric(
                "End-to-end pipeline",
                f"{col}_mean",
                float(vals.mean()),
                str(ARTIFACTS["pipeline_frames"])
            )

consolidated = pd.DataFrame(rows)
if not consolidated.empty:
    display(consolidated)
else:
    print("No consolidated metrics available yet.")

consolidated.to_csv(OUT / "final_consolidated_metrics.csv", index=False)

## 13. Generate final evaluation plots

In [ ]:
import matplotlib.pyplot as plt

# Plot condition distribution if predictions contain a ground-truth column.
if condition_predictions is not None and not condition_predictions.empty:
    candidate_true = [c for c in condition_predictions.columns
                      if c.lower() in {"true_condition", "ground_truth", "target", "label", "actual_condition"}]
    if candidate_true:
        c = candidate_true[0]
        counts = condition_predictions[c].value_counts()
        plt.figure(figsize=(8,4))
        counts.plot(kind="bar")
        plt.title("Condition Test-Set Ground-Truth Distribution")
        plt.xlabel("Condition")
        plt.ylabel("Images")
        plt.tight_layout()
        plt.savefig(OUT / "condition_test_distribution.png", dpi=150)
        plt.show()

# Risk distribution.
if risk_levels is not None and not risk_levels.empty:
    if "risk_level" in risk_levels.columns and "frames" in risk_levels.columns:
        plt.figure(figsize=(7,4))
        plt.bar(risk_levels["risk_level"].astype(str), risk_levels["frames"])
        plt.title("Risk-Level Distribution")
        plt.xlabel("Risk level")
        plt.ylabel("Frames")
        plt.tight_layout()
        plt.savefig(OUT / "risk_level_distribution.png", dpi=150)
        plt.show()

# Pipeline risk over frame index.
if pipeline_frames is not None and not pipeline_frames.empty:
    if {"frame_index","frame_risk"}.issubset(pipeline_frames.columns):
        plt.figure(figsize=(10,4))
        plt.plot(pipeline_frames["frame_index"], pipeline_frames["frame_risk"])
        plt.title("End-to-End Frame Risk")
        plt.xlabel("Frame index")
        plt.ylabel("Risk score")
        plt.tight_layout()
        plt.savefig(OUT / "pipeline_frame_risk.png", dpi=150)
        plt.show()

## 14. Final evaluation report

In [ ]:
final_report = {
    "project": "Adaptive_Road_Vision",
    "evaluation_type": "final_consolidated_evaluation",
    "generated_by": "27_final_evaluation.ipynb",
    "artifact_status": {
        row["artifact"]: bool(row["exists"])
        for _, row in artifact_status.iterrows()
    },
    "selected_detector_record": selected_detector,
    "components_evaluated": [
        "dataset_integrity",
        "condition_classification",
        "enhancement",
        "object_detection",
        "original_vs_enhanced_detection",
        "bytetrack",
        "monocular_depth",
        "risk_engine",
        "end_to_end_pipeline",
    ],
    "output_directory": str(OUT),
    "interpretation_notes": [
        "Only metrics actually present in project artifacts are reported.",
        "No missing metric is replaced with an invented value.",
        "Detector comparison is descriptive; detector selection follows the saved selection artifact when present.",
        "ByteTrack evaluation is operational because object-level tracking ground truth is not available in the current project metadata.",
        "Depth estimates are relative unless explicit calibration is provided.",
        "Enhancement PSNR/SSIM use pseudo-reference targets defined by the enhancement evaluation workflow.",
        "Risk score is a project-defined perception-risk score and should not be interpreted as collision probability.",
        "Runtime/FPS is reported only when present in the saved pipeline artifacts; results depend on hardware and input mode.",
    ],
}

# Add selected scalar summaries when available.
if not consolidated.empty:
    final_report["consolidated_metric_count"] = int(len(consolidated))

if pipeline_report is not None:
    final_report["pipeline_report"] = pipeline_report

if isinstance(detector_selection, dict):
    final_report["detector_selection_artifact"] = detector_selection

with open(OUT / "final_evaluation_report.json", "w", encoding="utf-8") as f:
    json.dump(final_report, f, indent=2, default=str)

print("Saved:", OUT / "final_evaluation_report.json")

## 15. Final project checklist

The final report should be considered **artifact-complete** only for components whose corresponding evaluation files exist.

### Recommended evidence chain

| Stage | Evidence |
|---|---|
| Dataset | Final dataset summary + split integrity |
| Condition | Test accuracy / balanced accuracy / macro-F1 + confusion matrix |
| Enhancement | Validation/test pseudo-reference PSNR/SSIM + downstream detection |
| Detection | YOLO11n vs YOLO11s comparison + saved selection artifact |
| Tracking | ByteTrack operational continuity diagnostics |
| Depth | Object-level relative depth estimates |
| Risk | Frame/object risk distributions |
| Integration | End-to-end pipeline CSV + runtime report |
| Final | `final_evaluation_report.json` |

A missing artifact is a **missing evaluation result**, not a zero score.

### Final caveats

This notebook is intended for the project's final experimental evaluation and documentation.

It should **not** claim:
- calibrated metric distance without calibration,
- MOTA/IDF1/HOTA without tracking ground truth,
- photographic enhancement quality from pseudo-reference metrics alone,
- collision probability from the risk score,
- or real-time deployment solely from a measured notebook FPS.

Use the component-specific notebooks alongside this final consolidation when presenting detailed methodology and results.